# ANPR — Indian Plate Detector Training (Colab)

**Tonight's goal:** a trained plate detector, exported to ONNX, downloaded, in about 60 minutes.

Run the cells top to bottom. Only **Cell 3 (CONFIG)** needs editing.

| Section | What it does | Time |
|---|---|---|
| 1–2 | GPU check + install | 2 min |
| 3–5 | Config + get the data (Kaggle or Roboflow) | 8 min |
| 6–7 | Auto-convert annotations to YOLO format | 2 min |
| 8 | Build the 2000-image subset + 100-image holdout | 1 min |
| 9 | **Train** | 30–45 min |
| 10–11 | Validate + export ONNX | 3 min |
| 12 | Verify ONNX matches PyTorch | 1 min |
| 13–14 | Generate demo plate list + download everything | 5 min |

## 1. Check the GPU

If this says 'No GPU', go to **Runtime → Change runtime type → T4 GPU** and re-run.

In [ ]:
import subprocess, sys

out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if out.returncode != 0:
    print('NO GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
else:
    print(out.stdout.split('\n')[8])
    print(out.stdout.split('\n')[9])

## 2. Install

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxslim
import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. CONFIG — edit this cell only

Set `SUBSET_SIZE` lower (e.g. 1000) if you are short on time.

In [ ]:
# ---------------- EDIT ME ----------------
SUBSET_SIZE   = 2500     # capped automatically if fewer are available
HOLDOUT_SIZE  = 100      # never seen by the model -- your demo set
EPOCHS        = 60
IMGSZ         = 640      # drop to 416 to roughly halve training time
BATCH         = 16
MODEL         = 'yolo11n.pt'
SEED          = 42
# -----------------------------------------

VAL_FRACTION = 0.15
WORK = '/content/anpr'
import os, random
random.seed(SEED)
os.makedirs(WORK, exist_ok=True)
print(f'{SUBSET_SIZE} images | {EPOCHS} epochs | {IMGSZ}px')

## 4. Get the data

Two paths. **Path A** = Indian plates (Kaggle). **Path B** = 10k general plates (Roboflow, faster).

If Path A gives you trouble, just use Path B and move on — the detector learns
"high-contrast rectangle on a vehicle", which transfers across countries.

---

### Path A — Kaggle (Indian plates)

Kaggle now uses a `KGAT_...` token string, **not** a kaggle.json file.
Get it at: kaggle.com → Settings → API → **Generate New Token** → copy the string.

In [ ]:
import os

# PASTE YOUR TOKEN HERE (starts with KGAT_)
KAGGLE_TOKEN = 'KGAT_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

!pip install -q --upgrade kaggle

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(KAGGLE_TOKEN.strip())
os.chmod('/root/.kaggle/access_token', 0o600)

print('testing auth...')
rc = os.system('kaggle datasets list -s "license plate" > /tmp/kt.txt 2>&1')
out = open('/tmp/kt.txt').read()
if rc == 0 and 'ref' in out.lower():
    print('AUTH OK -- run the download cell below')
    print(out[:400])
else:
    print('AUTH FAILED:')
    print(out[:600])
    print()
    print('-> Try Legacy API Credentials on the same Kaggle page (gives kaggle.json),')
    print('   or skip to Path B (Roboflow) below.')

**If auth failed and you got a legacy `kaggle.json` instead**, run this cell instead
of the one above:

In [ ]:
# from google.colab import files
# files.upload()                      # pick kaggle.json
# !pip install -q kaggle
# !mkdir -p /root/.kaggle && cp kaggle.json /root/.kaggle/
# !chmod 600 /root/.kaggle/kaggle.json
# !kaggle datasets list -s "license plate" | head -5

### Download the Indian datasets

In [ ]:
INDIAN_SETS = [
    'kedarsai/indian-license-plates-with-labels',   # ~2000 imgs, YOLO labels (best one)
    'saisirishan/indian-vehicle-dataset',           # Indian vehicles + plate annotations
    'dataclusterlabs/indian-number-plates-dataset', # multi-state Indian plates
]

import shutil
from pathlib import Path

RAW = Path(WORK) / 'raw'
shutil.rmtree(RAW, ignore_errors=True)
RAW.mkdir(parents=True)

for i, slug in enumerate(INDIAN_SETS):
    dest = RAW / f'ds{i}'
    dest.mkdir()
    print(f'--- {slug}')
    r = os.system(f'kaggle datasets download -d {slug} -p "{dest}" --unzip -q')
    n = len(list(dest.rglob('*.jpg'))) + len(list(dest.rglob('*.png')))
    if r != 0 or n == 0:
        print('    failed or empty -- skipping')
        shutil.rmtree(dest, ignore_errors=True)
    else:
        print(f'    ok, {n} images')

total = len(list(RAW.rglob('*.jpg'))) + len(list(RAW.rglob('*.png')))
print()
print(f'TOTAL: {total} images')
if total < 300:
    print('Too few -- run Path B below to top up.')

---

### Path B — Roboflow (10k plates, no file needed)

Free key at roboflow.com → sign in → Settings → Roboflow API → Private API Key.

Run this **instead of** Path A, or **as well as** it to top up. Both write into the same
folder and get merged automatically.

In [ ]:
from pathlib import Path
import shutil

RAW = Path(WORK) / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

!pip install -q roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_KEY")
rf.workspace("roboflow-universe-projects") \
  .project("license-plate-recognition-rxg4e") \
  .version(4).download("yolov8", location=str(RAW / 'ds_rf'))

total = len(list(RAW.rglob('*.jpg'))) + len(list(RAW.rglob('*.png')))
print(f'TOTAL now: {total} images')

### Path C — upload your own zip (optional)

In [ ]:
# from google.colab import files
# up = files.upload()
# name = list(up.keys())[0]
# !unzip -q "{name}" -d {WORK}/raw/ds_mine

## 5. Inspect what we got

Finds every image and every annotation file, whatever the folder layout.

In [ ]:
from pathlib import Path
from collections import Counter

RAW = Path(WORK) / 'raw'

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
ANN_EXT = {'.txt', '.xml', '.json'}

images = [p for p in RAW.rglob('*') if p.suffix.lower() in IMG_EXT]
anns   = [p for p in RAW.rglob('*') if p.suffix.lower() in ANN_EXT
          and p.name.lower() not in ('readme.txt', 'classes.txt', 'requirements.txt')]

print(f'images      : {len(images)}')
print(f'annotations : {len(anns)}')
print()
print('annotation types:', Counter(p.suffix.lower() for p in anns))
print()
print('sample image paths:')
for p in images[:3]:
    print('  ', p.relative_to(RAW))
print('sample annotation paths:')
for p in anns[:3]:
    print('  ', p.relative_to(RAW))

assert len(images) > 0, 'No images found. Check the unzip worked.'

## 6. Auto-detect the annotation format

Handles YOLO txt, Pascal VOC xml, 4-point polygon txt, and COCO json.

In [ ]:
import json as _json
import xml.etree.ElementTree as ET

def sniff_format(ann_paths):
    for p in ann_paths[:40]:
        s = p.suffix.lower()
        if s == '.xml':
            return 'voc'
        if s == '.json':
            try:
                d = _json.loads(p.read_text())
                if isinstance(d, dict) and 'annotations' in d and 'images' in d:
                    return 'coco'
            except Exception:
                pass
        if s == '.txt':
            for line in p.read_text().strip().split('\n'):
                parts = line.split()
                if len(parts) == 5:
                    try:
                        vals = [float(v) for v in parts[1:]]
                        if all(0.0 <= v <= 1.0 for v in vals):
                            return 'yolo'
                    except ValueError:
                        pass
                if len(parts) >= 8:
                    return 'poly4'
    return 'unknown'

FMT = sniff_format(anns)
print('detected format:', FMT)

if FMT == 'unknown':
    print()
    print('Could not auto-detect. Here is a sample annotation file:')
    print('-' * 50)
    print(anns[0].read_text()[:500])
    print('-' * 50)
    print('Set FMT manually below and re-run: FMT = "yolo" | "voc" | "poly4" | "coco"')

## 7. Convert everything to YOLO format

One class: `plate`.

In [ ]:
import cv2, shutil

STAGE = Path(WORK) / 'staged'
shutil.rmtree(STAGE, ignore_errors=True)
(STAGE / 'images').mkdir(parents=True)
(STAGE / 'labels').mkdir(parents=True)

img_by_stem = {}
for p in images:
    img_by_stem.setdefault(p.stem, p)


def to_yolo_line(x_min, y_min, x_max, y_max, w, h):
    xc = ((x_min + x_max) / 2) / w
    yc = ((y_min + y_max) / 2) / h
    bw = (x_max - x_min) / w
    bh = (y_max - y_min) / h
    if bw <= 0 or bh <= 0:
        return None
    xc, yc = min(max(xc, 0), 1), min(max(yc, 0), 1)
    bw, bh = min(bw, 1), min(bh, 1)
    return f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}'


def convert_one(ann_path):
    # returns (image_path, [yolo_lines]) or None
    stem = ann_path.stem
    img_path = img_by_stem.get(stem)
    if img_path is None:
        return None

    lines = []

    if FMT == 'yolo':
        for ln in ann_path.read_text().strip().split('\n'):
            parts = ln.split()
            if len(parts) == 5:
                lines.append('0 ' + ' '.join(parts[1:]))

    elif FMT == 'voc':
        try:
            root = ET.parse(ann_path).getroot()
        except Exception:
            return None
        size = root.find('size')
        w = int(float(size.find('width').text))
        h = int(float(size.find('height').text))
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            ln = to_yolo_line(float(bb.find('xmin').text), float(bb.find('ymin').text),
                              float(bb.find('xmax').text), float(bb.find('ymax').text), w, h)
            if ln:
                lines.append(ln)

    elif FMT == 'poly4':
        im = cv2.imread(str(img_path))
        if im is None:
            return None
        h, w = im.shape[:2]
        for ln in ann_path.read_text().strip().split('\n'):
            parts = ln.replace(',', ' ').split()
            nums = []
            for tok in parts:
                try:
                    nums.append(float(tok))
                except ValueError:
                    pass
            if len(nums) >= 8:
                xs, ys = nums[0:8:2], nums[1:8:2]
                out = to_yolo_line(min(xs), min(ys), max(xs), max(ys), w, h)
                if out:
                    lines.append(out)

    return (img_path, lines) if lines else None


pairs = []
if FMT == 'coco':
    coco_file = [p for p in anns if p.suffix == '.json'][0]
    d = _json.loads(coco_file.read_text())
    dims = {im['id']: (im['file_name'], im['width'], im['height']) for im in d['images']}
    grouped = {}
    for a in d['annotations']:
        grouped.setdefault(a['image_id'], []).append(a['bbox'])
    for img_id, boxes in grouped.items():
        fname, w, h = dims[img_id]
        ip = img_by_stem.get(Path(fname).stem)
        if ip is None:
            continue
        lns = []
        for x, y, bw, bh in boxes:
            ln = to_yolo_line(x, y, x + bw, y + bh, w, h)
            if ln:
                lns.append(ln)
        if lns:
            pairs.append((ip, lns))
else:
    for ap in anns:
        r = convert_one(ap)
        if r:
            pairs.append(r)

print(f'converted {len(pairs)} image/label pairs')
assert len(pairs) > 50, 'Too few pairs converted -- check FMT in cell 6.'

for ip, lns in pairs:
    shutil.copy(ip, STAGE / 'images' / f'{ip.stem}.jpg')
    (STAGE / 'labels' / f'{ip.stem}.txt').write_text('\n'.join(lns))

print('staged at', STAGE)

## 8. Build the subset + holdout

`SUBSET_SIZE` images for train/val, plus `HOLDOUT_SIZE` images the model **never sees**.
Those holdout images are your demo set.

In [ ]:
DS = Path(WORK) / 'subset'
shutil.rmtree(DS, ignore_errors=True)
for split in ['train', 'val']:
    (DS / split / 'images').mkdir(parents=True)
    (DS / split / 'labels').mkdir(parents=True)
HOLD = Path(WORK) / 'holdout'
HOLD.mkdir(parents=True, exist_ok=True)

stems = sorted(p.stem for p in (STAGE / 'images').glob('*.jpg'))
random.shuffle(stems)

need = min(SUBSET_SIZE + HOLDOUT_SIZE, len(stems))
picked = stems[:need]

n_hold = min(HOLDOUT_SIZE, len(picked) // 5)
holdout = picked[:n_hold]
rest    = picked[n_hold:]
n_val   = max(1, int(len(rest) * VAL_FRACTION))
val, train = rest[:n_val], rest[n_val:]

def place(stem_list, split):
    for s in stem_list:
        shutil.copy(STAGE / 'images' / f'{s}.jpg', DS / split / 'images' / f'{s}.jpg')
        shutil.copy(STAGE / 'labels' / f'{s}.txt', DS / split / 'labels' / f'{s}.txt')

place(train, 'train')
place(val, 'val')
for s in holdout:
    shutil.copy(STAGE / 'images' / f'{s}.jpg', HOLD / f'{s}.jpg')

print(f'train   : {len(train)}')
print(f'val     : {len(val)}')
print(f'holdout : {len(holdout)}   <- demo set, never trained on')

In [ ]:
yaml_text = f'''path: {DS}
train: train/images
val: val/images

nc: 1
names:
  0: plate
'''
yaml_path = Path(WORK) / 'subset.yaml'
yaml_path.write_text(yaml_text)
print(yaml_text)

### Sanity check — look at 4 training images with their boxes

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, s in zip(axes.flat, train[:4]):
    im = cv2.imread(str(DS / 'train' / 'images' / f'{s}.jpg'))
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for ln in (DS / 'train' / 'labels' / f'{s}.txt').read_text().strip().split('\n'):
        _, xc, yc, bw, bh = map(float, ln.split())
        x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 3)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()

> **Stop and look at those boxes.** If they are not tightly around the number plates,
> the conversion in cell 7 picked the wrong format. Fix it now — training on bad labels
> wastes the whole run.

## 9. Train

30–45 min on a T4. Leave it running and go build the backend.

In [ ]:
from ultralytics import YOLO
import time

model = YOLO(MODEL)

t0 = time.time()
model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=0.001,          # low LR because we start from pretrained weights
    lrf=0.01,
    warmup_epochs=3,
    patience=15,
    device=0,
    amp=True,
    workers=4,
    project=WORK,
    name='detector',
    exist_ok=True,
    # augmentation
    fliplr=0.5,
    flipud=0.0,         # never flip plates vertically
    degrees=10,
    perspective=0.0005,
    hsv_v=0.5,          # brightness variation -- helps with night shots
    mosaic=1.0,
)
print(f'\ndone in {(time.time() - t0) / 60:.1f} min')

## 10. Validate

In [ ]:
best = Path(WORK) / 'detector' / 'weights' / 'best.pt'
print('weights:', best, '|', round(best.stat().st_size / 1e6, 1), 'MB')

m = YOLO(str(best))
res = m.val(data=str(yaml_path), imgsz=IMGSZ, device=0)

print()
print(f'mAP@50    : {res.box.map50:.4f}')
print(f'mAP@50-95 : {res.box.map:.4f}')
print(f'precision : {res.box.mp:.4f}')
print(f'recall    : {res.box.mr:.4f}')
print()
if res.box.map50 > 0.85:
    print('GOOD -- demo ready.')
elif res.box.map50 > 0.70:
    print('USABLE -- demo will work, curate your demo images carefully.')
else:
    print('WEAK -- check the label boxes from cell 8, or train more epochs.')

### Look at predictions on the holdout set

In [ ]:
hold_imgs = sorted(HOLD.glob('*.jpg'))[:6]
preds = m.predict([str(p) for p in hold_imgs], imgsz=IMGSZ, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axes.flat, preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.show()

## 11. Export to ONNX

ONNX lets the backend run without PyTorch installed (~50 MB instead of ~2.5 GB).

In [ ]:
onnx_path = m.export(format='onnx', imgsz=IMGSZ, simplify=True, opset=12, dynamic=False)
onnx_path = Path(onnx_path)
print('exported:', onnx_path, '|', round(onnx_path.stat().st_size / 1e6, 1), 'MB')

## 12. Verify ONNX matches PyTorch

**Do not skip this.** ONNX export sometimes breaks silently, and finding out on your
laptop at 3am is miserable.

In [ ]:
import numpy as np, onnxruntime as ort

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name

def prep(path):
    im = cv2.imread(str(path))
    im = cv2.resize(im, (IMGSZ, IMGSZ))
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.transpose(im, (2, 0, 1))[None, ...]

ok = 0
test = sorted(HOLD.glob('*.jpg'))[:10]
for p in test:
    torch_n = len(m.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False)[0].boxes)
    out = sess.run(None, {inp_name: prep(p)})[0]
    scores = out[0, 4, :] if out.shape[1] < out.shape[2] else out[0, :, 4]
    onnx_n = int((scores > 0.25).sum())
    match = abs(torch_n - onnx_n) <= 2
    ok += match
    print(f'{p.name[:28]:30s} torch={torch_n}  onnx~={onnx_n}  {"OK" if match else "MISMATCH"}')

print()
print(f'{ok}/{len(test)} consistent')
print('PASS -- safe to download.' if ok >= len(test) - 2 else 'FAIL -- re-export with opset=11.')

## 13. Generate the demo plate list

Runs EasyOCR over the holdout detections and writes `demo_plates.csv`.
That CSV is what you use to write your registry seed script — saves you reading
100 plates by hand tonight.

In [ ]:
!pip install -q easyocr
import easyocr, csv, re

reader = easyocr.Reader(['en'], gpu=True)
PATTERN = re.compile(r'^[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{4}$')

rows = []
for p in sorted(HOLD.glob('*.jpg')):
    r = m.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    if not len(r.boxes):
        continue
    im = cv2.imread(str(p))
    b = r.boxes[0]
    x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
    crop = im[max(y1, 0):y2, max(x1, 0):x2]
    if crop.size == 0:
        continue
    out = reader.readtext(crop, detail=1)
    text = ''.join(o[1] for o in out).upper().replace(' ', '').replace('-', '')
    text = re.sub(r'[^A-Z0-9]', '', text)
    conf = sum(o[2] for o in out) / len(out) if out else 0.0
    rows.append({
        'image': p.name,
        'plate': text,
        'ocr_conf': round(float(conf), 3),
        'det_conf': round(float(b.conf[0]), 3),
        'valid_format': bool(PATTERN.match(text)),
    })

csv_path = Path(WORK) / 'demo_plates.csv'
with open(csv_path, 'w', newline='') as f:
    wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    wr.writeheader(); wr.writerows(rows)

good = [r for r in rows if r['valid_format']]
print(f'{len(rows)} plates read | {len(good)} match the Indian plate format')
print()
for r in good[:15]:
    print(f"  {r['plate']:14s} ocr={r['ocr_conf']:.2f}  {r['image']}")

### Auto-generate the seed script

In [ ]:
seed = ['# scripts/seed_demo.py  -- generated from demo_plates.csv',
        '# GRANTED / DENIED / UNKNOWN mix for the demo',
        '',
        'DEMO_VEHICLES = [']

names = ['R. Menon', 'S. Kumar', 'A. Rahman', 'P. Nair', 'D. Sharma',
         'V. Iyer', 'M. Das', 'K. Reddy', 'T. Joseph', 'N. Gupta']

for i, r in enumerate(good[:12]):
    if i < 8:
        auth, tag = 'True ', 'GRANTED'
    elif i < 10:
        auth, tag = 'False', 'DENIED'
    else:
        continue          # leave the rest out -> UNKNOWN
    seed.append(f'    ("{r["plate"]}", "{names[i % len(names)]}", "car", {auth}),   # {tag}')

seed += ['    # plates deliberately NOT registered -> UNKNOWN:']
for r in good[10:14]:
    seed.append(f'    #   {r["plate"]}')
seed += [']']

txt = '\n'.join(seed)
(Path(WORK) / 'seed_demo.py').write_text(txt)
print(txt)

## 14. Package and download

In [ ]:
BUNDLE = Path(WORK) / 'anpr_bundle'
shutil.rmtree(BUNDLE, ignore_errors=True)
(BUNDLE / 'weights').mkdir(parents=True)
(BUNDLE / 'demo_frames').mkdir(parents=True)

shutil.copy(onnx_path, BUNDLE / 'weights' / 'plate_detector.onnx')
shutil.copy(best,      BUNDLE / 'weights' / 'plate_detector.pt')
shutil.copy(csv_path,  BUNDLE / 'demo_plates.csv')
shutil.copy(Path(WORK) / 'seed_demo.py', BUNDLE / 'seed_demo.py')

for p in sorted(HOLD.glob('*.jpg')):
    shutil.copy(p, BUNDLE / 'demo_frames' / p.name)

(BUNDLE / 'model_info.txt').write_text(f'''ANPR plate detector
-------------------
architecture : {MODEL}
input size   : {IMGSZ}x{IMGSZ}
classes      : 1 (plate)
trained on   : {len(train)} images ({EPOCHS} epochs)
mAP@50       : {res.box.map50:.4f}
mAP@50-95    : {res.box.map:.4f}
precision    : {res.box.mp:.4f}
recall       : {res.box.mr:.4f}

demo_frames/ = {len(list(HOLD.glob("*.jpg")))} held-out images (never trained on)

Backend usage:
  import onnxruntime as ort
  sess = ort.InferenceSession("weights/plate_detector.onnx",
                              providers=["CPUExecutionProvider"])
  # input : float32 [1,3,{IMGSZ},{IMGSZ}], RGB, /255, NCHW
  # output: [1,5,N] -> cx, cy, w, h, conf   (apply NMS)
''')

shutil.make_archive(str(Path(WORK) / 'anpr_bundle'), 'zip', BUNDLE)
size = (Path(WORK) / 'anpr_bundle.zip').stat().st_size / 1e6
print(f'anpr_bundle.zip  {size:.1f} MB')
!ls -R {BUNDLE} | head -20

In [ ]:
from google.colab import files
files.download(f'{WORK}/anpr_bundle.zip')

### Also save to Drive (recommended — browser downloads fail sometimes)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(f'{WORK}/anpr_bundle.zip', '/content/drive/MyDrive/anpr_bundle.zip')
# print('saved to Drive')

---

## What you now have

```
anpr_bundle/
├── weights/
│   ├── plate_detector.onnx    <- backend/app/weights/
│   └── plate_detector.pt      <- backup
├── demo_frames/               <- backend/demo_frames/  (100 unseen images)
├── demo_plates.csv            <- what the OCR read from each
├── seed_demo.py               <- ready-made registry seed
└── model_info.txt             <- metrics + I/O spec for the backend
```

**Next:** unzip into your project, then ask for the Claude Code master prompt.